## Train / Test Split
Split the FITS files 80/20 and make matching CSVs.

In [ ]:
import os, glob, shutil
import pandas as pd
from sklearn.model_selection import train_test_split

fits_folder   = "fits"
csv_file      = "coordinates.csv"
train_folder  = "train"
test_folder   = "test"
test_size     = 0.2
random_state  = 42

os.makedirs(train_folder, exist_ok=True)
os.makedirs(test_folder,  exist_ok=True)

fits_files = sorted(glob.glob(os.path.join(fits_folder, "*.fits")))
print(f"Total FITS files: {len(fits_files)}")

train_files, test_files = train_test_split(
    fits_files, test_size=test_size, random_state=random_state
)
print(f"Train: {len(train_files)} files")
print(f"Test:  {len(test_files)} files")


In [ ]:
print("Copying files...")
for i, f in enumerate(train_files, 1):
    shutil.copy(f, os.path.join(train_folder, os.path.basename(f)))
    if i % 50 == 0:
        print(f"  Train: {i}/{len(train_files)}")

for i, f in enumerate(test_files, 1):
    shutil.copy(f, os.path.join(test_folder, os.path.basename(f)))
    if i % 50 == 0:
        print(f"  Test: {i}/{len(test_files)}")


In [ ]:
df = pd.read_csv(csv_file, skiprows=[1])
print(f"Total CSV rows: {len(df)}")

def fname_to_id(path):
    return os.path.basename(path).replace('.fits','').lower().replace('_','')

train_ids = [fname_to_id(f) for f in train_files]
test_ids  = [fname_to_id(f) for f in test_files]

print(f"Sample train IDs: {train_ids[:3]}")
print(f"Sample CSV Data IDs: {df['Data ID'].head(3).tolist()}")

train_df = df[df['Data ID'].isin(train_ids)].copy()
test_df  = df[df['Data ID'].isin(test_ids)].copy()

train_df.to_csv("train.csv", index=False)
test_df.to_csv("test.csv",  index=False)

print(f"Train CSV: {len(train_df)} rows saved to train.csv")
print(f"Test CSV:  {len(test_df)} rows saved to test.csv")

if len(train_df) == 0 or len(test_df) == 0:
    print("WARNING: Empty CSV – check if filenames match Data IDs")

print("Done!")
